# BehaviorGPT Showcase

**BehaviorGPT** is trained on long sequences of things people actually did (viewed this, added that, bought the other) and predicts what comes next. Same idea as a language model predicting the next word, except the sequence is actions instead of text.

This notebook walks through the model in eight steps. Steps 1 to 4 use a sample Amazon catalog. Steps 5 to 8 embed your own catalog and are optional.

## Step 1. Setup

You need an API key. Create yours up at [unboxai.com/behaviorgpt](https://unboxai.com/behaviorgpt); the key also arrives by email. Put it in a `.env` file at the repo root as `UNBOXAI_EMBED_API_KEY` (see `.env.example`).

The client packages your event history into HTTP requests. `market` sets the locale and `default_catalog_id` picks the catalog every call uses unless told otherwise. Here we browse a sample e-commerce catalog as a user from the US.

In [ ]:
from behaviorgpt import UnboxAIClient, Search, View, AddToCart, Order

# To load API Key from .env
from dotenv import load_dotenv
load_dotenv(override=True);

In [ ]:
client = UnboxAIClient(market="us", default_catalog_id="amazon_catalog")

## Step 2. First query

`complete` takes a history of events and returns the products most likely to come next. Here the history is a single search for "lego".

In [ ]:
res = client.complete(history=[Search("lego")], limit=5)
res.to_pandas()[["name", "price", "score"]]

`score` is the model's confidence in each product. Higher means more likely to come next.

## Step 3. One call, many features

`complete` predicts what comes next in a sequence of events, the way a language model completes a sentence. The same call covers every feature in a store; only the history differs:

- **Cold start:** empty history. No personalization signal at all.
- **Search:** the last event is a query.
- **More like this:** the last event is a product.
- **Personalized search:** a product, then a query.
- **Post-purchase:** a product viewed, added to cart, and ordered.

The catalog is the model's vocabulary: the complete list of things it can return. Nothing outside it ever comes back. The ids below are known products in the sample catalog.

In [ ]:
# Example known product_ids from the catalog in use by default

NIKE_PANTS = "B08NYK61PJ"

EXPENSIVE_SNEAKERS_1 = "B09DBFS1Y5"
EXPENSIVE_SNEAKERS_2 = "B0BS1YV27T"

CHEAP_SNEAKERS_1 = "B0CNFSW297"
CHEAP_SNEAKERS_2 = "B08P34GCHQ"

In [ ]:
scenarios = {
    "cold start":          [],
    "search":              [Search("shoes")],
    "more like this":      [View(NIKE_PANTS)],
    "personalized search": [View(NIKE_PANTS), Search("shoes")],
    "post-purchase":       [View(NIKE_PANTS), AddToCart(NIKE_PANTS), Order()],
}

for label, history in scenarios.items():
    top = client.complete(history=history, limit=3)
    print(f"{label:22} -> {', '.join(top.names)}")

## Step 4. Context changes the answer

The model reads the whole sequence, not just the last thing typed. Same final query, two different preceding events:

In [ ]:
res_one = client.complete(history=[Search("nike"), Search("shoes")], limit=5)
res_two = client.complete(history=[Search("adidas"), Search("shoes")], limit=5)

display(res_one.to_pandas()[["name", "price", "score"]], res_two.to_pandas()[["name", "price", "score"]])

The word "shoes" did not change. What came before it did.

The next cell pushes further: the same query after clicks on expensive versus cheap sneakers. Nothing about price is passed to the model.

In [ ]:
QUERY = "sneakers"

for label, history in {
    "no history": [],
    "expensive":  [View(EXPENSIVE_SNEAKERS_1), AddToCart(EXPENSIVE_SNEAKERS_1), View(EXPENSIVE_SNEAKERS_2), AddToCart(EXPENSIVE_SNEAKERS_2)],
    "budget":     [View(CHEAP_SNEAKERS_1), AddToCart(CHEAP_SNEAKERS_1), View(CHEAP_SNEAKERS_2), AddToCart(CHEAP_SNEAKERS_2)],
}.items():
    res = client.complete(history=[*history, Search(QUERY)], catalog_id="amazon_catalog", limit=5)

    print(f"{label:12} avg ${res.mean_price:>6.0f}  ->  {', '.join(res.names[:5])}")

The model picked up price sensitivity from the clicks alone.

## Step 5. Bring your own catalog (optional)

The model can only return products from its vocabulary, so to use it on your store you embed your catalog. Structure a parquet file as described in [docs/catalog-format.md](../docs/catalog-format.md), set `CATALOG_PATH` to it, and run the cell.

`embed` uploads the file and starts a job: images are fetched, products are embedded, results are indexed. With `wait=True` the call blocks until the job finishes (a few minutes). Without it, the call returns immediately and queries return `status` until processing completes. The returned `catalog_id` is private to your API key. `catalog_name` is your file name.

In [ ]:
CATALOG_PATH = "./sample_catalog.parquet"

job = client.embed(CATALOG_PATH, wait=True, timeout=1800.0)
job.model_dump()

## Step 6. Inspect the space

Each product is now a vector. Similar products land close together, unrelated ones far apart, and search and recommendation become nearest-neighbor lookups. The plot below projects the space to two dimensions.

In [ ]:
from IPython.display import HTML
HTML(client.umap(catalog_id=job.catalog_id))

Check the space by finding products similar to a random one. To test a specific product, replace `pick.id` with its id as a string.

In [ ]:
pick = client.random_product(job.catalog_id)
print(pick.data["name"])

response = client.similar_products(pick.id, catalog_id=job.catalog_id)
response.to_pandas()[["name", "price", "score"]]

## Step 7. Query your catalog

Same calls as steps 2 and 3, now with `catalog_id=job.catalog_id`. Change the queries to match what your catalog sells.

In [ ]:
res = client.complete(history=[Search("lego sets")], catalog_id=job.catalog_id, limit=5)
res.to_pandas()[["name", "price", "score"]]

In [ ]:
SOME_PRODUCT = res.products.items[0].data["id"]

scenarios = {
    "cold start":          [],
    "search":              [Search("kitchen")],
    "more like this":      [View(SOME_PRODUCT)],
    "personalized search": [View(SOME_PRODUCT), Search("kitchen")],
}

for label, history in scenarios.items():
    top = client.complete(catalog_id=job.catalog_id, history=history, limit=3)
    names = [item.data.get("name") for item in top.products.items]
    print(f"{label:22} -> {', '.join(names)}")

## Step 8. Try it in the demo

Once the catalog is embedded, use it in the interactive demo with BehaviorGPT as the recommendation engine:

1. Open [behaviorgpt.unboxai.com](https://behaviorgpt.unboxai.com/).
2. Select the **BehaviorGPT V4.0-13B** model.
3. In the catalog dropdown, choose **Bring your own catalog** and enter your API key.

Product images must be publicly reachable for the demo to display them.